# Lekce 12 - Redukce historie chatu pomocí Agent Scratchpadu

Tento notebook ukazuje, jak spravovat kontext v dlouhých konverzacích pomocí Microsoft Agent Framework. Jak konverzace rostou, počet tokenů se zvyšuje — nakonec překročí okno kontextu modelu. Tento problém řešíme pomocí **vzorce shrnutí kontextu** a **agent scratchpadu** pro přetrvávající paměť.

## Co se naučíte:
1. **Proč je správa kontextu důležitá**: Pochopení limitů tokenů a kontextových oken
2. **Agenti vnímaví k kontextu**: Vytváření agentů, kteří spravují svůj vlastní kontext konverzace
3. **Vzor shrnutí kontextu**: Použití nástrojů ke zhuštění historie konverzace
4. **Agent Scratchpad**: Přetrvávající paměť, která přežívá redukci kontextu

## Předpoklady:
- Nastavení Azure OpenAI s nakonfigurovanými environmentálními proměnnými
- Pochopení základních konceptů agentů z předchozích lekcí


## Nastavení


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Proč je správa kontextu důležitá

Každý LLM má omezené **okno kontextu** — maximální počet tokenů, které může zpracovat v jednom požadavku. Jak se konverzace s více koly vyvíjí:

- **Počet tokenů roste lineárně** s každou uživatelskou zprávou a odpovědí asistenta.
- **Tokeny v promptu dominují nákladům**, protože celá historie je odesílána znovu při každém kole.
- Nakonec konverzace **překročí okno kontextu** a model buď ořízne data, nebo dojde k chybě.

### Strategie správy kontextu

| Strategie | Jak funguje | Kompromis |
|---|---|---|
| **Oříznutí** | Odstranění nejstarších zpráv | Ztráta raného kontextu |
| **Shrnutí** | Zestručnění starších zpráv do shrnutí | Některé detaily ztraceny, ale klíčové body zachovány |
| **Scratchpad / Externí paměť** | Ukládání klíčových faktů mimo konverzaci | Vyžaduje volání nástrojů, ale přetrvává i po redukci |

V tomto notebooku kombinujeme **shrnutí** s nástrojem **scratchpad**, aby agent mohl zachovat kontinuitu i když je historie konverzace zhuštěná.


## Vytváření agenta s povědomím o kontextu


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulace dlouhého rozhovoru

Projděme si vícetahový rozhovor, abychom viděli, jak se kontext hromadí. Agent by měl po celou dobu uchovávat klíčové detaily (preference, rozpočet, data cesty) a ukázat kontinuitu.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Všimněte si, jak si agent uchovává kontext z předchozích kol — ví o Japonsku, sushi, chrámech, fotografii, rozpočtu 3000 dolarů, cestování o samotě a období v dubnu. V krátké konverzaci to funguje dobře, ale jak konverzace roste, je drahé opakovaně posílat celý předešlý obsah.

Pojďme pokračovat v konverzaci s dalšími koly, abychom viděli, jak se kontext hromadí:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Vzor souhrnu kontextu

S růstem konverzace můžeme použít **nástroj pro shrnutí** k zhuštění nasbíraného kontextu do kompaktní podoby. Agent tento nástroj vyvolá, aby zaznamenal klíčová přání, takže i když jsou starší zprávy odstraněny, podstatné informace zůstanou zachovány.

Tento vzor je stavebním kamenem pro sofistikovanější zkrácení historie:
1. Agent identifikuje klíčová fakta z konverzace
2. Zavolá nástroj pro shrnutí, aby je uložil
3. Starší zprávy mohou být bezpečně odstraněny, protože souhrn zachycuje to podstatné

Níže definujeme nástroj `summarize_preferences`, který může agent použít k zaznamenání kompaktního shrnutí toho, co se naučil.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Shrnutí

V této lekci jste se naučili, jak spravovat kontext v dlouhotrvajících agentních konverzacích pomocí Microsoft Agent Framework:

### Klíčové koncepty
- **Kontextová okna jsou omezená** — každý token v historii konverzace něco stojí a započítává se do limitu.
- **Nástroje pro shrnutí** umožňují agentovi zhušťovat nahromaděný kontext do kompaktních shrnutí, čímž snižují používání tokenů a zároveň zachovávají podstatné informace.
- **Agentní poznámkové bloky** poskytují trvalou externí paměť, která přežije jakékoliv zmenšení konverzace.

### Co jste vytvořili
- **Agent s povědomím o kontextu**, který udržuje kontinuitu napříč vícestupňovými konverzacemi
- **Nástroj pro shrnutí** (`summarize_preferences`), který zaznamenává klíčové uživatelské údaje v kompaktním formátu
- **Vícekolová konverzace** demonstrující udržení kontextu a řešení změn

### Aplikace v reálném světě
- **Boti zákaznické podpory**: Pamatují si preference během dlouhých podporovacích relací
- **Osobní asistenti**: Sledují probíhající projekty bez nutnosti znovu vysvětlovat kontext
- **Vzdělávací lektoři**: Udržují pokrok studentů přes mnoho interakcí

### Další kroky
- Implementovat plnohodnotný poznámkový blok s perzistencí založenou na souborech
- Přidat automatické zkracování historie po shrnutí
- Kombinovat s vektorovými databázemi pro sémantické vyhledávání v paměti
- Vytvořit agenty, kteří mohou obnovit konverzace po několika dnech s plným kontextem


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Upozornění**:  
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho původním jazyce je považován za autoritativní zdroj. Pro zásadní informace doporučujeme profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění či mylné výklady vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
